# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Adtividad en Equipos Semanas 7 y 8 : LDA y LMM audio-a-texto**


| Nombre | Matricula |
|-----|-----|
| Fernando Arango Gaviria | A01797660 |
| Erick Alan Cuéllar Quintanilla | A01383577 |
| María de los Angeles Rabelero Campos | A01793031 |
| Luis Pablo Pérez Esmeral | A01420939 |


**Número de Equipo:**. Equipo 37

* ##### **En cada ejercicio pueden importar los paquetes o librerías que requieran.**

* ##### **En cada ejercicio pueden incluir las celdas y líneas de código que deseen.**

In [1]:
!pip install gensim --upgrade --user

In [2]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...
import os
import re
import pandas as pd

import torch #Para uso del GPU

import librosa   # paquete para audio y música
from transformers import pipeline
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq #Para usar el modelo de Whisper

import warnings
# para filtrar advertencias:
warnings.filterwarnings("ignore")


import nltk
from nltk.corpus import stopwords

# Para el algoritmo LDA
import gensim
from gensim import corpora


# **Ejercicio 1:**

* #### **Liga de los audios de las fábulas de Esopo:** https://www.gutenberg.org/ebooks/21144

* #### **Descargar los 10 archivos de audio solicitados: 1, 4, 5, 6, 14, 22, 24, 25, 26, 27.**



In [3]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

import requests

# Diccionario con las fábulas y sus enlaces directos en MP3
fabulas = {
    "fabula1": "https://www.gutenberg.org/files/21144/mp3/21144-01.mp3",
    "fabula4": "https://www.gutenberg.org/files/21144/mp3/21144-04.mp3",
    "fabula5": "https://www.gutenberg.org/files/21144/mp3/21144-05.mp3",
    "fabula6": "https://www.gutenberg.org/files/21144/mp3/21144-06.mp3",
    "fabula14": "https://www.gutenberg.org/files/21144/mp3/21144-14.mp3",
    "fabula22": "https://www.gutenberg.org/files/21144/mp3/21144-22.mp3",
    "fabula24": "https://www.gutenberg.org/files/21144/mp3/21144-24.mp3",
    "fabula25": "https://www.gutenberg.org/files/21144/mp3/21144-25.mp3",
    "fabula26": "https://www.gutenberg.org/files/21144/mp3/21144-26.mp3",
    "fabula27": "https://www.gutenberg.org/files/21144/mp3/21144-27.mp3"
}

# Descargar y guardar cada archivo
for nombre, url in fabulas.items():
    print(f"Descargando {nombre} desde {url}...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"{nombre}.mp3", "wb") as f:
            f.write(response.content)
        print(f"{nombre} descargada correctamente.")
    else:
        print(f"Error al descargar {nombre}: {response.status_code}")



Descargando fabula1 desde https://www.gutenberg.org/files/21144/mp3/21144-01.mp3...
fabula1 descargada correctamente.
Descargando fabula4 desde https://www.gutenberg.org/files/21144/mp3/21144-04.mp3...
fabula4 descargada correctamente.
Descargando fabula5 desde https://www.gutenberg.org/files/21144/mp3/21144-05.mp3...
fabula5 descargada correctamente.
Descargando fabula6 desde https://www.gutenberg.org/files/21144/mp3/21144-06.mp3...
fabula6 descargada correctamente.
Descargando fabula14 desde https://www.gutenberg.org/files/21144/mp3/21144-14.mp3...
fabula14 descargada correctamente.
Descargando fabula22 desde https://www.gutenberg.org/files/21144/mp3/21144-22.mp3...
fabula22 descargada correctamente.
Descargando fabula24 desde https://www.gutenberg.org/files/21144/mp3/21144-24.mp3...
fabula24 descargada correctamente.
Descargando fabula25 desde https://www.gutenberg.org/files/21144/mp3/21144-25.mp3...
fabula25 descargada correctamente.
Descargando fabula26 desde https://www.gutenberg

# **Ejercicio 2a:**

* #### **Comenten el por qué del modelo seleccionado para extracción del texto de los audios.**

* #### **Extraer el contenido de los audios en texto.**

* #### **Sugerencia:** pueden extraerlo en un formato de diccionario, clave:valor $→$ {audio01:fabula01, ...}

In [4]:
# Con librosa podemos obtener información del archivo:

fabulas = ['fabula1', 'fabula4', 'fabula5', 'fabula6', 'fabula14', 'fabula22', 'fabula24',
           'fabula25', 'fabula26', 'fabula27']

for nombre in fabulas:
  audio_path = f"/content/{nombre}.mp3"
  audio, sample_rate = librosa.load(audio_path)
  print(f"Duración del audio de {nombre}: {len(audio)/sample_rate:.1f} segundos")

Duración del audio de fabula1: 72.6 segundos
Duración del audio de fabula4: 67.1 segundos
Duración del audio de fabula5: 58.4 segundos
Duración del audio de fabula6: 71.8 segundos
Duración del audio de fabula14: 58.1 segundos
Duración del audio de fabula22: 57.2 segundos
Duración del audio de fabula24: 64.5 segundos
Duración del audio de fabula25: 56.5 segundos
Duración del audio de fabula26: 63.7 segundos
Duración del audio de fabula27: 60.3 segundos


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


El modelo seleccionado es **openai/whisper-large-v3-turbo** por que es la version de Whisper de OPENAI que tiene alta precisión vs sus modelos mas pequeños, maneja bien las diferentes calidades de audio, pero a su vez, es optimizado para trabajar mas rapido que el modelo large-v3.

Parte de su precisión se pudo comprobar con el lenguaje en español has identificar perfectamente diferentes palabras, acentos y caracteres especiales propios del idioma.

Esto nos ayudo a garantizar que todo el ejercicio completo (Del audio hasta el análisis del LLM) se desarrollara correctamente y con datos confiables.

In [6]:
model_id = "openai/whisper-large-v3-turbo"    # una versión ligera de v3

# Cargamos el modelo:
processor = AutoProcessor.from_pretrained(model_id)
modelo = AutoModelForSpeechSeq2Seq.from_pretrained(model_id).to(device)

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.71M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

In [7]:
# Inicializamos el pipeline:
pipe = pipeline("automatic-speech-recognition",
                model=modelo,
                tokenizer=processor.tokenizer,
                feature_extractor=processor.feature_extractor,
                dtype=torch.float32,      #torch.float16 if torch.cuda.is_available() else torch.float32,
                device=device
                )

In [8]:
# <Opcional: si deseas escucharlo>
from IPython.display import Audio
Audio(audio_path)

In [9]:
registros = []
for nombre in fabulas:
  audio_path = f"/content/{nombre}.mp3"
  audio, sample_rate = librosa.load(audio_path)
  print(f"Obteniendo texto de {nombre}")

  result = pipe(audio,
                return_timestamps=True,   # Para audios mayores a 30 segs.
                generate_kwargs={"language":"es"}  # Lo detecta automático, pero se le puede indicar el idioma.
                )
  # Guardar registro
  registros.append({"titulo": nombre,
                "transcripcion": result["text"]})
  print(f"Texto guardado de {nombre}")

# Crear DataFrame
df = pd.DataFrame(registros)

Obteniendo texto de fabula1


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

Texto guardado de fabula1
Obteniendo texto de fabula4
Texto guardado de fabula4
Obteniendo texto de fabula5
Texto guardado de fabula5
Obteniendo texto de fabula6
Texto guardado de fabula6
Obteniendo texto de fabula14
Texto guardado de fabula14
Obteniendo texto de fabula22
Texto guardado de fabula22
Obteniendo texto de fabula24
Texto guardado de fabula24
Obteniendo texto de fabula25
Texto guardado de fabula25
Obteniendo texto de fabula26


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Texto guardado de fabula26
Obteniendo texto de fabula27
Texto guardado de fabula27


# **Ejercicio 2b:**

* #### **Eliminar el inicio y final comunes de los textos extraídos de cada fábula.**

* #### **Sugerencia:** Pueden guardar esta información en un archivo tipo JSON, para que al estar probando diferentes opciones en los ejercicios siguientes, puedan recuperar rápidamente la información de cada video/fábula.

In [10]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

def limpiar_texto(texto):

    # Se elimina la introducción (Todo lo que este antes de "número XX" con ., o espacio
    texto = re.sub(r".*?número\s*\d+[., ]*", "", texto, flags=re.IGNORECASE | re.DOTALL)

    # Se elimina el final de la fábula (desde "Fin de la fábula" hasta el final del texto)
    texto = re.sub(r"Fin de la fábula.*", "", texto, flags=re.IGNORECASE | re.DOTALL)
    texto = re.sub(r"Fin de fábula.*", "", texto, flags=re.IGNORECASE | re.DOTALL)

    # Se quitan espacios extra
    return texto.strip()

# Aplicar limpieza a todo el DataFrame
df["texto_limpio"] = df["transcripcion"].apply(limpiar_texto)

print(df[["transcripcion", "texto_limpio"]])



                                       transcripcion  \
0   Las Fábulas de Sopo Grabado para LibriVox.org...   
1   Las Fábulas de Esopo, grabado para LibriVox.o...   
2   Las Fábulas de Sopo, grabado para LibriVox.or...   
3   Las fábulas de Esopo, grabado para LibriVox.o...   
4   Las Fábulas de Esopo Grabado para LibriVox.or...   
5   Las Fábulas de Esopo Grabado para LibriVox.or...   
6   Las fábulas de Sopo, grabado para LibriVox.or...   
7   Las fábulas de Esopo. Grabado para LibreVox.o...   
8   Las Fábulas de Esopo Grabado para LibriVox.or...   
9   Las Fábulas de Esopo Grabado para LibriVox.or...   

                                        texto_limpio  
0  El Lobo y el Cordero en el Templo Dándose cuen...  
1  El Lobo y la Coruja A un lobo que comía un hue...  
2  El Lobo y el Caballo. Pasaba un lobo por un se...  
3  El Lobo y el Asno. Un lobo fue elegido rey ent...  
4  El Lobo y el Cabrito Encerrado Protegido por l...  
5  El perro y la almeja Un perro de esos acostumb... 

In [11]:
df

,titulo,transcripcion,texto_limpio
0,fabula1,Las Fábulas de Sopo Grabado para LibriVox.org...,El Lobo y el Cordero en el Templo Dándose cuen...
1,fabula4,"Las Fábulas de Esopo, grabado para LibriVox.o...",El Lobo y la Coruja A un lobo que comía un hue...
2,fabula5,"Las Fábulas de Sopo, grabado para LibriVox.or...",El Lobo y el Caballo. Pasaba un lobo por un se...
3,fabula6,"Las fábulas de Esopo, grabado para LibriVox.o...",El Lobo y el Asno. Un lobo fue elegido rey ent...
4,fabula14,Las Fábulas de Esopo Grabado para LibriVox.or...,El Lobo y el Cabrito Encerrado Protegido por l...
5,fabula22,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro y la almeja Un perro de esos acostumb...
6,fabula24,"Las fábulas de Sopo, grabado para LibriVox.or...",El perro y el reflejo en el río. Badeaba un pe...
7,fabula25,Las fábulas de Esopo. Grabado para LibreVox.o...,El perro y el carnicero. Penetró un perro en u...
8,fabula26,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro con campanilla Había un perro que aco...
9,fabula27,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro que perseguía al león Un perro de cas...


# **Ejercicio 3:**

* #### **Apliquen el proceso de limpieza que consideren adecuado.**

* #### **Justifiquen los pasos de limpieza utilizados. Tomen en cuenta que el texto extraído de cada fábula es relativamente pequeño.**

* #### **En caso de que decidan no aplicar esta etapa de limpieza, deberán justificarlo.**

El preprocesamiento es una etapa crítica para que se tenga un diccionario con calidad; por lo tanto se realizaron limpiezas necesarias.

*   Eliminación de stopwords que no agregan significado relevante al texto.
*   Conversión a minúsculas para homogenizar las palabras.
* Eliminación de simbolos o números que no aportan valor al análisis de las fabulas.
* Manejo de caracteres especiales del idioma español, como ñ o vocales con acento.
* Filtro de tokens de una sola letra que no aportan valor relevante.

Al tratarse de textos cortos, se decidio no generar filtros por palabras con poca frecuencia.

In [12]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

# Descargar stopwords en español
nltk.download('stopwords')

# Lista de stopwords en español
stopwords_es = stopwords.words('spanish')

# Palabras de negación que NO queremos eliminar
negwords_es = [
    "no", "nunca", "jamás", "nadie", "ninguno", "ninguna",
    "ni", "tampoco", "sin"
]

# Crear lista personalizada de stopwords en español
mystopwords_es = [w for w in stopwords_es if w not in negwords_es]

print("Ejemplo de stopwords en español:", mystopwords_es[:30])


Ejemplo de stopwords en español: ['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le', 'ya', 'o', 'este', 'sí', 'porque', 'esta']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
df

,titulo,transcripcion,texto_limpio
0,fabula1,Las Fábulas de Sopo Grabado para LibriVox.org...,El Lobo y el Cordero en el Templo Dándose cuen...
1,fabula4,"Las Fábulas de Esopo, grabado para LibriVox.o...",El Lobo y la Coruja A un lobo que comía un hue...
2,fabula5,"Las Fábulas de Sopo, grabado para LibriVox.or...",El Lobo y el Caballo. Pasaba un lobo por un se...
3,fabula6,"Las fábulas de Esopo, grabado para LibriVox.o...",El Lobo y el Asno. Un lobo fue elegido rey ent...
4,fabula14,Las Fábulas de Esopo Grabado para LibriVox.or...,El Lobo y el Cabrito Encerrado Protegido por l...
5,fabula22,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro y la almeja Un perro de esos acostumb...
6,fabula24,"Las fábulas de Sopo, grabado para LibriVox.or...",El perro y el reflejo en el río. Badeaba un pe...
7,fabula25,Las fábulas de Esopo. Grabado para LibreVox.o...,El perro y el carnicero. Penetró un perro en u...
8,fabula26,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro con campanilla Había un perro que aco...
9,fabula27,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro que perseguía al león Un perro de cas...


In [14]:
#Se crea la función para la limpieza
def clean_tok(doc):

  Texto = str(doc)

  #Los procesos de limpieza son:

  #Volver todo el texto a minusculas
  Texto = Texto.lower()

  #Considerar solo caracteres alfabeticos.  Se eliminan todas los otros simbolos o caracteres numericos (Como esta en ingles el texto, dejamos solamente a-z sin caracteres especiales como ñ)
  Texto = re.sub(r'[^a-záéíóúüñ]',' ',Texto)

  #Tokenizaión - Se separa el texto por palabras
  tokens = Texto.split()

  #Se eliminan las Stopwords y se valida que las palabras tengan una longitud mayor a 1
  tokens = [token for token in tokens if token not in mystopwords_es and len(token)>1]

  return tokens

# Aplicar la función a cada fila de la columna 'texto_limpio'
df["tokens"] = df["texto_limpio"].apply(clean_tok)

# Ver resultado
print(df[["titulo", "tokens"]].head())

     titulo                                             tokens
0   fabula1  [lobo, cordero, templo, dándose, cuenta, perse...
1   fabula4  [lobo, coruja, lobo, comía, hueso, atragantó, ...
2   fabula5  [lobo, caballo, pasaba, lobo, sembrado, cebada...
3   fabula6  [lobo, asno, lobo, elegido, rey, congéneres, d...
4  fabula14  [lobo, cabrito, encerrado, protegido, segurida...


In [15]:
df

,titulo,transcripcion,texto_limpio,tokens
0,fabula1,Las Fábulas de Sopo Grabado para LibriVox.org...,El Lobo y el Cordero en el Templo Dándose cuen...,"[lobo, cordero, templo, dándose, cuenta, perse..."
1,fabula4,"Las Fábulas de Esopo, grabado para LibriVox.o...",El Lobo y la Coruja A un lobo que comía un hue...,"[lobo, coruja, lobo, comía, hueso, atragantó, ..."
2,fabula5,"Las Fábulas de Sopo, grabado para LibriVox.or...",El Lobo y el Caballo. Pasaba un lobo por un se...,"[lobo, caballo, pasaba, lobo, sembrado, cebada..."
3,fabula6,"Las fábulas de Esopo, grabado para LibriVox.o...",El Lobo y el Asno. Un lobo fue elegido rey ent...,"[lobo, asno, lobo, elegido, rey, congéneres, d..."
4,fabula14,Las Fábulas de Esopo Grabado para LibriVox.or...,El Lobo y el Cabrito Encerrado Protegido por l...,"[lobo, cabrito, encerrado, protegido, segurida..."
5,fabula22,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro y la almeja Un perro de esos acostumb...,"[perro, almeja, perro, acostumbrados, comer, h..."
6,fabula24,"Las fábulas de Sopo, grabado para LibriVox.or...",El perro y el reflejo en el río. Badeaba un pe...,"[perro, reflejo, río, badeaba, perro, río, lle..."
7,fabula25,Las fábulas de Esopo. Grabado para LibreVox.o...,El perro y el carnicero. Penetró un perro en u...,"[perro, carnicero, penetró, perro, carnicería,..."
8,fabula26,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro con campanilla Había un perro que aco...,"[perro, campanilla, perro, acostumbraba, morde..."
9,fabula27,Las Fábulas de Esopo Grabado para LibriVox.or...,El perro que perseguía al león Un perro de cas...,"[perro, perseguía, león, perro, casa, encontró..."


# **Ejercicio 4:**

In [16]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

# Se crear el diccionario a partir de todos los tokens
dictionary = corpora.Dictionary(df["tokens"])

# Convertir cada fábula en bolsa de palabras (BoW)
corpus = [dictionary.doc2bow(text) for text in df["tokens"]]

In [17]:
# Siempre podemos expresar cada documento con palabras para un mejor entendimiento:
[[(dictionary[id], fre) for id, fre in cor] for cor in corpus[:2]]

[[('adentro', 1),
  ('allí', 1),
  ('así', 1),
  ('cercano', 1),
  ('colmillos', 1),
  ('corderito', 1),
  ('cordero', 2),
  ('cuenta', 1),
  ('decidió', 1),
  ('dijo', 1),
  ('dios', 2),
  ('dándose', 1),
  ('encontraba', 1),
  ('honor', 1),
  ('inmolaría', 1),
  ('llamó', 1),
  ('lobo', 3),
  ('mayor', 1),
  ('mejor', 1),
  ('pequeño', 1),
  ('perecer', 1),
  ('perseguido', 1),
  ('prefiero', 1),
  ('refugiarse', 1),
  ('remedio', 1),
  ('replicó', 1),
  ('sacrificador', 1),
  ('sacrificados', 1),
  ('ser', 2),
  ('si', 2),
  ('sin', 1),
  ('templo', 2),
  ('tener', 1),
  ('vale', 1),
  ('vamos', 1),
  ('víctima', 1)],
 [('dijo', 1),
  ('lobo', 4),
  ('si', 1),
  ('aceptó', 1),
  ('amiga', 1),
  ('aquella', 1),
  ('atragantó', 1),
  ('atravesado', 1),
  ('auxilio', 1),
  ('boca', 2),
  ('busca', 1),
  ('cabeza', 2),
  ('cancelación', 1),
  ('comía', 1),
  ('convenida', 1),
  ('correr', 1),
  ('corruptos', 1),
  ('corría', 1),
  ('coruja', 1),
  ('crees', 1),
  ('dejan', 1),
  ('ello'

In [18]:
# Entrenar modelo LDA (un tópico por documento)
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=10,       # un tema por fábula
    passes=20,          # número de iteraciones
    alpha=0.2,        # alfa: similaridad document-topics
    eta=0.2,         # beta: similaridad topic-words
    random_state=42
)

In [19]:
# Extraer palabras clave para cada fábula

def obtener_palabras_clave(doc_bow, num_palabras=20):
    # Obtener distribución de tópicos para el documento
    topics = lda_model.get_document_topics(doc_bow)

    # Tomar el tópico más probable
    topico_id = max(topics, key=lambda x: x[1])[0]

    # Extraer las palabras clave del tópico
    palabras = lda_model.show_topic(topico_id, topn=num_palabras)
    return [palabra for palabra, _ in palabras]

# Aplicar a cada fábula
df["palabras_clave"] = [obtener_palabras_clave(doc_bow) for doc_bow in corpus]

# Mostrar resultados
for i, row in df.iterrows():
    print(f"Fábula: {row['titulo']}")
    print("Palabras clave:", row["palabras_clave"])
    print("-"*50)

Fábula: fabula1
Palabras clave: ['lobo', 'si', 'dios', 'ser', 'templo', 'cordero', 'tener', 'sin', 'dijo', 'encontraba', 'mejor', 'perseguido', 'mayor', 'sacrificados', 'pequeño', 'dándose', 'remedio', 'decidió', 'colmillos', 'replicó']
--------------------------------------------------
Fábula: fabula4
Palabras clave: ['campanilla', 'lobo', 'hueso', 'paga', 'dijo', 'boca', 'cabeza', 'garganta', 'grulla', 'pidió', 'no', 'perro', 'hagas', 'nunca', 'partes', 'corría', 'atravesado', 'comía', 'todas', 'atragantó']
--------------------------------------------------
Fábula: fabula5
Palabras clave: ['no', 'cebada', 'caballo', 'lobo', 'encontró', 'amigo', 'camino', 'lobos', 'pasaba', 'vez', 'comieran', 'repuso', 'malvado', 'comida', 'cantidad', 'preferido', 'creérsele', 'rato', 'llevó', 'oídos']
--------------------------------------------------
Fábula: fabula6
Palabras clave: ['león', 'perro', 'lobo', 'dijo', 'primero', 'casa', 'asno', 'ley', 'no', 'partió', 'mismo', 'rápidamente', 'siempre', 

# **Ejercicio 5a y 5b:**

* #### **5a: Mediante el LLM que hayan seleccionado, generar un único enunciado que describa o resuma cada fábula.**

* #### **5b: Mediante el LLM que hayan seleccionado, generar tres posibles enunciados diferentes relacionados con la historia de la fábula.**

* #### **Sugerencia:** En realidad los dos incisos a y b se pueden obtener con un solo prompt que solicite la información y el formato correspondiente para cada una de estas partes. Por ejemplo, para cada fábula la salida puede ser un primer enunciado genérico que resume o describe dicha temática; seguido de tres enunciados, cada uno hablando sobre una situación o parte diferente de la fábula.

In [20]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

from google.colab import userdata
my_sk = userdata.get('SecretKey_OpenAI')

from openai import OpenAI
client = OpenAI(api_key=my_sk)

#Activar Modelo
response = client.responses.create(
    model="gpt-4o-mini",
    input="Write a one-sentence bedtime story about a unicorn."
)
print(response.output_text)



As the moonlight shimmered on the peaceful meadow, a gentle unicorn named Luna spread her wings and soared into the starry sky, painting dreams for all the children below.


In [21]:
#Create the User promt y el system prompt

for i, row in df.iterrows():
  titulo = row['titulo']
  palabras_clave = row['palabras_clave']

  user_prompt = (f"Usando estas palabras clave: {', '.join(palabras_clave)}\n"
        f"""Genera un único enunciado que resuma la fábula titulada '{titulo}'.
        Este enunciado no debe ser mayor a 15 palabras\n"""
        f"Además, genera 3 posibles subtemas enunciados diferentes."
        f"Al inicio mostrar el titulo para tener claridad de que fábula se trata"
    )

  system_prompt = """Eres experto en literatura y se van a analizar las palabras clave de algunas fábulas."""

  response_temas = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
  )

  #Return the assistant's answer
  print(f"\n{response_temas.choices[0].message.content}")


**Fábula1**  
Enunciado resumen:  
El lobo decidió ser mejor sacrificando sus colmillos, sin remedio para el cordero.  

Subtemas:  
1. La lucha entre la fuerza y la inocencia en la naturaleza.  
2. El sacrificio voluntario para evitar la violencia.  
3. La búsqueda de coexistencia entre depredador y presa.

Fábula4:  
Enunciado resumen:  
El perro con hueso en boca ignoró la advertencia y se atragantó.

Subtemas:  
1. La codicia puede llevar a consecuencias peligrosas.  
2. Escuchar consejos previene daños innecesarios.  
3. La importancia de no confiar en las apariencias.

Fábula5:  
El lobo malvado no creyó al caballo y trató de comerse su comida.

Subtemas posibles:  
1. La importancia de la confianza entre amigos en el camino.  
2. Cómo el engaño del lobo casi llevó a la pérdida.  
3. La vigilancia ante la amenaza de lobos malvados.

Fábula6:  
El león dijo que el listo asno siempre supera imprevistos rápidamente en la misma empresa.

Subtemas:  
1. La importancia de la astucia f

# **Ejercicio 6:**

* #### **Incluyan sus conclusiones de la actividad audio-a-texto:**

Este ejercicio nos permitió recorrer todo el flujo de trabajo de un proyecto de procesamiento de lenguaje natural:

Conversión de audio a texto  
El primer reto fue seleccionar y configurar el modelo adecuado para transcribir las grabaciones. En nuestro caso se utilizó Whisper. Vimos como la elección del modelo y el formato del audio impactan directamente en la calidad del texto inicial.

Luego en la parte de preprocesamiento y construcción de diccionarios, la limpieza del texto es crítica. Se eliminaron las introducciones y los cierres utilizando regex, donde se tuvieron que hacer muchas pruebas para lograr el resultado esperado.  Se limpió el texto y se utilizaron stopwords en español, donde tuvimos que revisar con cuidado el uso de caracteres especiales como la ñ y las vocales acentuadas.

Uno de los puntos más delicados fue configurar correctamente la regex y los filtros para que el pipeline respetara las particularidades del español. Este paso costó trabajo porque un error podía eliminar palabras clave o distorsionar el sentido de las frases.

Al aplicar el algoritmo LDA, vimos cómo cada fábula puede ser representada por un conjunto de palabras clave. Aquí tuvimos que tener cuidado con el número de tópicos con el que se entrena el modelo, para que los resultados fueran realmente representativos.

Al final, el uso de LLM para análisis de las palabras clave, nos permitieron transformar estas listas en resúmenes narrativos y subtemas alternativos, sin necesidad de procesar todo el documento completo. Esto es particularmente útil al momento de requerir  análisis eficientes y creativos, donde el LLM aporta contexto y estilo a partir de contextos reducidos.



# **Fin de la actividad LDA y LMM: audio-a-texto**